In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))  # Add parent directory to path

import numpy as np
from visualisation import *
from plotly.subplots import make_subplots
from figures.generate_data import load_image_point_cloud
from diffusion_geometry.classes.main import DiffusionGeometry


Create some data

In [ ]:
data1 = load_image_point_cloud("../data/data7.jpeg", n=400, threshold=0.9, intensity_weighted=True)

plot_scatter_2d(data1, color = 'black').show()
dg1 = DiffusionGeometry.from_point_cloud(data1)

In [ ]:
line_width=0.8

f = dg1.function(data1[:,1])
f11 = plot_scatter_2d(data1, color = f.to_ambient(), size=4)
f11.show()

h = dg1.function_space.zeros()
h.coeffs[1] = 1.0
X = h.grad()
f12 = plot_quiver_2d(data1, X.to_ambient(), scale = 0.05, line_width=line_width)
f12.show()

fa = X(f)
f13 = plot_scatter_2d(data1, color = fa.to_ambient(), size=4)
f13.show()

In [ ]:
h = dg1.function_space.zeros()
h.coeffs[4] = 1.0
f21 = plot_scatter_2d(data1, color = h.to_ambient(), size=4)
f21.show()

vortex = np.stack((-data1[:,1], data1[:,0] - 0.45), axis=1)
vortex /= np.linalg.norm(vortex, axis=1)[:, np.newaxis]**1.5
Y = dg1.form(vortex, degree=1).sharp()
f22 = plot_quiver_2d(data1, Y.to_ambient(), scale = 0.06, line_width=line_width)
f22.show()

Yh = Y(h)
f23 = plot_scatter_2d(data1, color = Yh.to_ambient(), size=4)
f23.show()

In [ ]:
k = dg2.function_space.zeros()
k.coeffs[2] = 1.0
f31 = plot_scatter_3d(data2, color = k.to_ambient(), size=2.5, camera=camera)
f31.show()

evals, evecs = dg2.laplacian(1).spectrum()
Z = evecs[0].sharp()
f32 = plot_quiver_3d(data2, Z.to_ambient(), scale = 0.15, line_width=1.4, camera=camera, arrow_scale=0.4)
f32.update_layout(scene_camera=camera)
f32.show()

Zk = Z(k)
f33 = plot_scatter_3d(data2, color = Zk.to_ambient(), size=2.5, camera=camera)
f33.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# --- Define subplot grid ---
specs = [
    [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
    [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
    [{"type": "scene"}, {"type": "scene"}, {"type": "scene"}],
]

fig = make_subplots(
    rows=3,
    cols=3,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.03,
    row_heights=[0.3, 0.3, 0.4],
)

# --- Add traces row by row ---
fig.add_traces(list(f11.data), rows=[1] * len(f11.data), cols=[1] * len(f11.data))
fig.add_traces(list(f12.data), rows=[1] * len(f12.data), cols=[2] * len(f12.data))
fig.add_traces(list(f13.data), rows=[1] * len(f13.data), cols=[3] * len(f13.data))

fig.add_traces(list(f21.data), rows=[2] * len(f21.data), cols=[1] * len(f21.data))
fig.add_traces(list(f22.data), rows=[2] * len(f22.data), cols=[2] * len(f22.data))
fig.add_traces(list(f23.data), rows=[2] * len(f23.data), cols=[3] * len(f23.data))

fig.add_traces(list(f31.data), rows=[3] * len(f31.data), cols=[1] * len(f31.data))
fig.add_traces(list(f32.data), rows=[3] * len(f32.data), cols=[2] * len(f32.data))
fig.add_traces(list(f33.data), rows=[3] * len(f33.data), cols=[3] * len(f33.data))

# --- Synchronise 2D axis ranges ---
all_2d = [f11, f12, f13, f21, f22, f23]
x_vals, y_vals = [], []
for f in all_2d:
    for t in f.data:
        if hasattr(t, "x") and t.x is not None:
            x_vals.extend([v for v in t.x if v is not None])
        if hasattr(t, "y") and t.y is not None:
            y_vals.extend([v for v in t.y if v is not None])
xrange = [min(x_vals), max(x_vals)]
yrange = [min(y_vals), max(y_vals)]
fig.update_xaxes(range=xrange, row=1)
fig.update_yaxes(range=yrange, row=1)
fig.update_xaxes(range=xrange, row=2)
fig.update_yaxes(range=yrange, row=2)

# --- Synchronise 3D ranges ---
all_3d = [f31, f32, f33]
x3d, y3d, z3d = [], [], []
for f in all_3d:
    for t in f.data:
        if hasattr(t, "x") and t.x is not None:
            x3d.extend(t.x)
        if hasattr(t, "y") and t.y is not None:
            y3d.extend(t.y)
        if hasattr(t, "z") and t.z is not None:
            z3d.extend(t.z)
xrange3 = [min(x3d), max(x3d)]
yrange3 = [min(y3d), max(y3d)]
zrange3 = [min(z3d), max(z3d)]
for i in range(1, 4):
    key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout(
        {
            key: dict(
                camera=camera,
                xaxis=dict(range=xrange3),
                yaxis=dict(range=yrange3),
                zaxis=dict(range=zrange3),
                aspectmode="data",
            )
        }
    )

# --- Final layout ---
fig.update_layout(
    width=1200,
    height=800,
    showlegend=False,
    margin=dict(l=0, r=0, t=0, b=0),
)


clean_fig(fig)
fig.show()
fig.write_image("figs/summary_3x2.pdf", scale=1)


In [ ]:
def labels(row, col):
    f = ['f', 'h', 'k'][row - 1]
    X = ['X', 'Y', 'Z'][row - 1]
    if col == 1:
        return rf"${f}$"
    elif col == 2:
        return rf"${X}$"
    elif col == 3:
        return rf"${X}({f})$"
    else:
        return ""

overpic_labels(fig, labels, 
            #    stretch_x=0.98, 
               stretch_y=1,
               offset_x=-1.0,
               offset_y=11.0)
